# Active-Learning Hyperspectral Invasive Species Segmentation via Adapted SAM 2
## Complete Google Colab Execution Notebook (Re-Run Suite)

This notebook runs the complete end-to-end AL-HSI-SAM2 experiment on a Google Colab T4 GPU (or A100/V100).
It is configured so you can **simply press "Run" cell-by-cell or select `Runtime > Run all`**.

### Workflow Overview:
- **Step 0**: GPU Check, Drive Mount, Git Sync & Dependencies
- **Step 0B**: Download Datasets & SAM 2.1 Checkpoint + Smoke Test
- **Step 0C**: Result Protection Safeguard (Automatic timestamped backup)
- **Step 1**: Re-Run Pavia Active Learning (BALD, Entropy, Random — with NDVI stall fix)
- **Step 2**: Run Indian Pines Full Benchmark (Adapter Training + AL Loop for 3 strategies)
- **Step 3**: Re-Run LoRA Rank Ablation Study (r=4, 8, 16 with checkpoint isolation fix)
- **Step 4**: Generate Web Dashboard Data (Pavia & Indian Pines GPU/CPU inference)
- **Step 5**: Download All Results (ZIP archives for dashboard & paper tables)
- **Step 6**: Local Dashboard Instructions
- **Appendix**: Optional Baseline Training & Full Ablations

### Step 0A: Verify GPU Accelerator
Ensure you have selected **Runtime > Change runtime type > T4 GPU** (or A100) in the top menu before proceeding.

In [ ]:
# Check GPU hardware
!nvidia-smi
import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: Running on CPU! Go to Runtime > Change runtime type > T4 GPU")

### Step 0B: Mount Google Drive & Environment Setup
Mounts Google Drive (if using Drive storage) or sets up the cloned repository, pulls the latest bug fixes from GitHub, and installs required dependencies.

In [ ]:
import os, sys

# 1. Mount Google Drive if running in Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        try:
            drive.mount('/content/drive')
        except Exception as e:
            print(f"Note: Drive mount skipped or already mounted: {e}")

# 2. Locate and navigate to project directory
possible_paths = [
    '/content/drive/MyDrive/CP',
    '/content/drive/MyDrive/hyperspectral-sam2-al',
    '/content/hyperspectral-sam2-al',
    os.getcwd()
]

project_dir = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, 'configs', 'default.yaml')):
        project_dir = p
        break

if project_dir:
    os.chdir(project_dir)
    print(f"✅ Working directory set to: {os.getcwd()}")
else:
    # If not found, clone repository
    print("Repository not found in Drive. Cloning from GitHub...")
    !git clone https://github.com/rushant22/hyperspectral-sam2-al.git
    %cd hyperspectral-sam2-al

# 3. Pull latest bug fixes (handles dirty tree safely)
!git stash
!git pull origin main

# 4. Install all dependencies
!pip install -q einops scikit-image pyyaml tqdm seaborn hydra-core omegaconf iopath scipy scikit-learn
!pip install -q git+https://github.com/facebookresearch/sam2.git
print("\n✅ Environment setup and dependencies ready!")

### Step 0C: Download Datasets & SAM 2.1 Checkpoint
Downloads the Pavia University and Indian Pines hyperspectral datasets, as well as the official Meta SAM 2.1 Hiera Base+ checkpoint (~309 MB).
> **Idempotent**: If files already exist in `./datasets/` or `./checkpoints/`, they will be automatically skipped without re-downloading.

In [ ]:
# Download datasets and SAM 2.1 checkpoint (skips if already present)
!python data/download.py --dataset all --sam2-checkpoint

# Run smoke test to verify all models and components
!python scripts/smoke_test.py

### Step 0D: Future-Run Safeguard (Automatic Result Backup)
> [!IMPORTANT]
> **Protects Previous Runs**: If `./results/` already contains completed experiment checkpoints (such as previous Result 3 data), this cell creates an automatic timestamped snapshot in `./results_archive_YYYYMMDD_HHMMSS/` so past results are never accidentally lost or overwritten.

In [ ]:
import os, shutil, time

results_dir = './results'
if os.path.exists(results_dir) and any(os.scandir(results_dir)):
    timestamp = time.strftime('%Y%m%d_%H%M%S')
    backup_dir = f'./results_archive_{timestamp}'
    shutil.copytree(results_dir, backup_dir)
    print(f"🛡️ Safety Archive Created: Backed up existing results to: {backup_dir}")
else:
    os.makedirs(results_dir, exist_ok=True)
    print("✅ Ready: Fresh results directory initialized.")

# Ensure Indian Pines output dir exists
os.makedirs('./results_indian_pines', exist_ok=True)

## Step 1: Re-Run Pavia Active Learning (with NDVI Stall Fix)
> **What was fixed**: The previous run had 0 new labels queried in rounds 7, 9, 10 due to an overly restrictive vegetation mask. In the fixed code, the querier falls back smoothly so every round gets real informative queries. Furthermore, checkpoints are saved per round (`al_bald_round01.pt` ... `al_bald_round10.pt`).

You can run each strategy individually below. If your session is interrupted, you can resume by running just the remaining cells.

In [ ]:
# Step 1A: Pavia Active Learning — BALD Strategy (Bayesian Active Learning by Disagreement)
print("=== Starting Pavia AL: BALD Strategy ===")
!python scripts/run_al_loop.py \
    --config configs/default.yaml \
    --strategy bald \
    --output_dir ./results

In [ ]:
# Step 1B: Pavia Active Learning — Shannon Entropy Strategy (Uncertainty Baseline)
print("=== Starting Pavia AL: Entropy Strategy ===")
!python scripts/run_al_loop.py \
    --config configs/default.yaml \
    --strategy entropy \
    --output_dir ./results

In [ ]:
# Step 1C: Pavia Active Learning — Random Query Strategy (Passive Learning Baseline)
print("=== Starting Pavia AL: Random Strategy ===")
!python scripts/run_al_loop.py \
    --config configs/default.yaml \
    --strategy random \
    --output_dir ./results

## Step 2: Run Indian Pines Full Benchmark (First Time Ever)
> **About Indian Pines**: AVIRIS sensor, 145×145 spatial dimensions, 200 spectral bands, 16 agricultural crop classes.
> Treated as a single patch without tiling; `ndvi_threshold: 0.0` ensures all crop land cover is active.
> Outputs are saved separately in `./results_indian_pines/`.

In [ ]:
# Step 2A: Train the Spectral Cross-Attention Adapter on Indian Pines
print("=== Training Adapted SAM 2 on Indian Pines ===")
!python scripts/train_adapter.py --config configs/indian_pines.yaml

In [ ]:
# Step 2B: Indian Pines AL — BALD Strategy
print("=== Starting Indian Pines AL: BALD Strategy ===")
!python scripts/run_al_loop.py \
    --config configs/indian_pines.yaml \
    --strategy bald \
    --output_dir ./results_indian_pines

In [ ]:
# Step 2C: Indian Pines AL — Entropy Strategy
print("=== Starting Indian Pines AL: Entropy Strategy ===")
!python scripts/run_al_loop.py \
    --config configs/indian_pines.yaml \
    --strategy entropy \
    --output_dir ./results_indian_pines

In [ ]:
# Step 2D: Indian Pines AL — Random Strategy
print("=== Starting Indian Pines AL: Random Strategy ===")
!python scripts/run_al_loop.py \
    --config configs/indian_pines.yaml \
    --strategy random \
    --output_dir ./results_indian_pines

## Step 3: Re-Run LoRA Rank Ablation Study (r = 4, 8, 16)
> **What was fixed**: Previous ablation runs showed identical scores for rank 4, 8, and 16 because weights were inadvertently retained across variants. The updated script deletes checkpoints between runs to guarantee isolated, independent evaluation.

In [ ]:
# Run LoRA rank ablation (r=4 vs r=8 vs r=16)
print("=== Running LoRA Rank Ablation Study ===")
!python scripts/run_ablations.py \
    --config configs/default.yaml \
    --ablation lora_rank

## Step 4: Generate Web Dashboard Data (GPU Inference)
Runs inference on both Pavia and Indian Pines datasets using the trained checkpoints, generating the 5 JSON files needed by the interactive web dashboard:
- `false_color.json`: False color PCA composite image
- `uncertainty_map.json`: Uncertainty heatmap
- `segmentation_map.json`: Ground truth and model predictions
- `query_history.json`: Active learning queried coordinate history per round
- `metrics_summary.json`: Per-round mIoU comparison across all strategies

In [ ]:
# Generate dashboard data for Pavia University
print("=== Generating Pavia Dashboard Data ===")
!python scripts/generate_dashboard_data.py \
    --config configs/default.yaml \
    --results_dir ./results \
    --output_dir ./dashboard_data_pavia

# Generate dashboard data for Indian Pines
print("\n=== Generating Indian Pines Dashboard Data ===")
!python scripts/generate_dashboard_data.py \
    --config configs/indian_pines.yaml \
    --results_dir ./results_indian_pines \
    --output_dir ./dashboard_data_indian_pines

## Step 5: Package & Download All Results
Packages the dashboard data and all experiment JSONs into zip archives and triggers browser downloads.

In [ ]:
import os, shutil
from google.colab import files

print("=== Creating Result Packages ===")

# 1. Pack Pavia dashboard data
if os.path.exists('./dashboard_data_pavia'):
    shutil.make_archive('dashboard_pavia', 'zip', './dashboard_data_pavia')
    files.download('dashboard_pavia.zip')
    print("✅ Queued dashboard_pavia.zip download")

# 2. Pack Indian Pines dashboard data
if os.path.exists('./dashboard_data_indian_pines'):
    shutil.make_archive('dashboard_indian_pines', 'zip', './dashboard_data_indian_pines')
    files.download('dashboard_indian_pines.zip')
    print("✅ Queued dashboard_indian_pines.zip download")

# 3. Pack key experiment JSONs for both datasets
os.makedirs('./results_export', exist_ok=True)
json_files = [
    'al_results_bald.json', 'al_results_entropy.json',
    'al_results_random.json', 'adapter_training_log.json',
    'ablation_results.json'
]

for src_dir, tag in [('./results', 'pavia'), ('./results_indian_pines', 'ip')]:
    if os.path.exists(src_dir):
        for f in json_files:
            src_path = os.path.join(src_dir, f)
            if os.path.exists(src_path):
                dest_name = f.replace('.json', f'_{tag}.json')
                shutil.copy(src_path, os.path.join('./results_export', dest_name))

if os.listdir('./results_export'):
    shutil.make_archive('all_results', 'zip', './results_export')
    files.download('all_results.zip')
    print("✅ Queued all_results.zip download")

print("\n🎉 All packages created and downloads initiated!")

## Step 6: Local Dashboard Setup Guide
Once the zip files are downloaded to your computer:

1. **Extract `dashboard_pavia.zip`**:
   Copy the extracted `.json` files into your project at:
   `visualization/dashboard/data/pavia/`

2. **Extract `dashboard_indian_pines.zip`**:
   Copy the extracted `.json` files into your project at:
   `visualization/dashboard/data/indian_pines/`

3. **Open the Dashboard**:
   From your project root, run:
   ```bash
   python -m http.server 8080 --directory visualization/dashboard
   ```
   Open `http://localhost:8080` in your browser. Use the dataset dropdown in the top header to toggle between **Pavia University** and **Indian Pines**!

---
## Appendix (Optional): Baseline Training & Full Ablations
The cells below are optional and provided if you wish to train the unadapted baseline model or run all ablations from scratch.

In [ ]:
# (Optional) Train unadapted baseline model on Pavia
# !python scripts/train_baseline.py --config configs/default.yaml

# (Optional) Run all ablation studies
# !python scripts/run_ablations.py --config configs/default.yaml